# U-Net input ablations: Ddc / full Yb / imaginary Yb / real Yb

Updated interface for `dl_normalized_datasets` v2. Set CASE=30 or 57. The terminal runner and notebook use the same code. Run from this code folder.


In [5]:
from pathlib import Path
import subprocess
import sys
import json
import csv

CODE_DIR = Path.cwd()  # Folder containing train_ablation.py and updated model files
DATA_ROOT = Path('/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset').resolve()  # EDIT if needed
CASE = 57              # 30, 39 or 57
TARGET = 'dq'          # 'dq' or 'vmag'; one target type per run
MODELS = ['unet', 'attention_unet']
VARIANTS = ['ddc', 'ddc_yb', 'ddc_imag', 'ddc_real']
SEEDS = [42, 123, 2026]
EPOCHS = 200
BATCH_SIZE = 64
LOSS_ALPHA = 0.5       # 1.0 means pure MSE
DEVICE = 'cuda'
SMOKE = False         # True uses tiny subsets/two epochs; not a performance estimate
RESUME = False
AMP = True

DATA_DIR = DATA_ROOT / f'case{CASE}'
OUT_DIR = CODE_DIR / ('smoke_results' if SMOKE else 'results') / f'case{CASE}_{TARGET}'
assert (CODE_DIR / 'train_ablation.py').is_file(), 'Set CODE_DIR to the extracted updated code folder'
for name in ['train', 'val', 'test']:
    assert (DATA_DIR / f'dataset_{name}.npz').is_file(), f'Missing {name} data'
print('Data:', DATA_DIR)
print('Results:', OUT_DIR)


Data: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case57
Results: /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/results/case57_dq


In [6]:
command = [sys.executable, '-u', str(CODE_DIR / 'train_ablation.py'),
    '--data-dir', str(DATA_DIR), '--out-dir', str(OUT_DIR),
    '--models', *MODELS, '--variants', *VARIANTS,
    '--target', TARGET, '--seeds', *map(str, SEEDS),
    '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
    '--loss-alpha', str(LOSS_ALPHA), '--device', DEVICE]
if SMOKE: command.append('--smoke')
if RESUME: command.append('--resume')
if not AMP: command.append('--no-amp')
print(' '.join(command))
subprocess.run(command, cwd=CODE_DIR, check=True)


/opt/tljh/user/bin/python -u /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/train_ablation.py --data-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case57 --out-dir /home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/results/case57_dq --models unet attention_unet --variants ddc ddc_yb ddc_imag ddc_real --target dq --seeds 42 123 2026 --epochs 200 --batch-size 64 --loss-alpha 0.5 --device cuda
PyTorch 2.10.0+cu128 device cuda
Dataset(dq, X=(6713, 3, 57, 57), y=(6713, 1, 57, 57)) 
 Dataset(dq, X=(1438, 3, 57, 57), y=(1438, 1, 57, 57))

unet ddc seed=42: 1,927,009 parameters; AMP=True
epoch=  1 loss=3.31258 val_R2=0.6760985556313879 val_RMSE=3.613537 (18.5s)
epoch=  2 loss=0.28082 val_R2=0.73050869809379 val_RMSE=3.296085 (1.5s)
epoch=  3 loss=0.23858 val_R2=0.796917793555253 val_RMSE=2.861291 (1.5s)
epoch=  4 loss=0.21746 val_R2=0.8081835235710131 val_RMSE=2.780795 (1.5s)
epoch=  5 loss=0.20509 val_R2=0.81663

CompletedProcess(args=['/opt/tljh/user/bin/python', '-u', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/train_ablation.py', '--data-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/dataset/case57', '--out-dir', '/home/jupyter-r.yazdi/myproject/dti3d/0pycodes/Shahr/DQ_dynamic_bus/Unet/results/case57_dq', '--models', 'unet', 'attention_unet', '--variants', 'ddc', 'ddc_yb', 'ddc_imag', 'ddc_real', '--target', 'dq', '--seeds', '42', '123', '2026', '--epochs', '200', '--batch-size', '64', '--loss-alpha', '0.5', '--device', 'cuda'], returncode=0)

In [7]:
with (OUT_DIR / 'summary.json').open() as f:
    summary = json.load(f)
print('Metrics use original units:', summary['protocol']['metric_units'])
for model, variants in summary['aggregate'].items():
    for variant, splits in variants.items():
        print(model, variant, 'test:', splits['test'])
print('Training-mean baseline:', summary['training_mean_baseline'])
print('Individual runs:', OUT_DIR / 'comparison.csv')


Metrics use original units: raw DQ
unet ddc test: {'r2': {'mean': 0.8500447486312828, 'std_across_seeds': 0.0014412778647418985}, 'rmse': {'mean': 2.461273936061629, 'std_across_seeds': 0.011811958362033343}, 'mae': {'mean': 0.9519006743329171, 'std_across_seeds': 0.017550874057290968}}
unet ddc_yb test: {'r2': {'mean': 0.8466810657822773, 'std_across_seeds': 0.0012167957067351633}, 'rmse': {'mean': 2.4887315840493587, 'std_across_seeds': 0.009884781773143015}, 'mae': {'mean': 1.0071337505281939, 'std_across_seeds': 0.021248926005021363}}
unet ddc_imag test: {'r2': {'mean': 0.847814627765788, 'std_across_seeds': 0.0019192546641638743}, 'rmse': {'mean': 2.479494387642735, 'std_across_seeds': 0.015658113187396995}, 'mae': {'mean': 0.9989169677026348, 'std_across_seeds': 0.03459743956686229}}
unet ddc_real test: {'r2': {'mean': 0.8479263662715399, 'std_across_seeds': 0.0033160795437986333}, 'rmse': {'mean': 2.4785183556607358, 'std_across_seeds': 0.027071172214491932}, 'mae': {'mean': 0.9

## Notes

All four variants share the same fitted training normalization, parameter count within each architecture, initial weights per seed, and training protocol. Excluded slots are masked after normalization. Test data is evaluated only after validation checkpoint selection. Do not apply a second scaler or sigmoid to standardized targets. The saved predictions and R2/RMSE/MAE are in original units. See README.md for inference and recovery.
